# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 19 - Dependency Parsing

### Demo mới:
- Chủ đề: Phân tích cú pháp phụ thuộc tiếng Việt
- Dữ liệu: Universal Dependencies Vietnamese VTB
- Mô hình: Stanza Vietnamese Dependency Parser

In [4]:
!pip install -q datasets stanza scikit-learn pandas

In [1]:
import pandas as pd
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import classification_report

# Tải bộ dữ liệu công khai Universal Dependencies Vietnamese VTB.
# Mỗi câu có token, POS tag, head index và nhãn dependency theo chuẩn UD.
print("Loading Universal Dependencies Vietnamese VTB...")
dataset = load_dataset("commul/universal_dependencies", "vi_vtb")

train_size = min(500, len(dataset["train"]))
test_size = min(100, len(dataset["test"]))

train_data = dataset["train"].select(range(train_size))
test_data = dataset["test"].select(range(test_size))

print(f"Available splits: {list(dataset.keys())}")
print(f"Total training sentences available: {len(dataset['train'])}")
print(f"Total testing sentences available: {len(dataset['test'])}")
print(f"Training sentences used: {len(train_data)}")
print(f"Testing sentences used: {len(test_data)}")

# Xem phân bố nhãn dependency trong tập train để thấy parser phải học nhiều loại quan hệ.
deprel_counts = Counter(label for example in train_data for label in example["deprel"])
deprel_distribution = pd.DataFrame({
    "Dependency Relation": list(deprel_counts.keys()),
    "Count": list(deprel_counts.values()),
}).sort_values("Count", ascending=False)

display(deprel_distribution.head(15))

sample = train_data[0]
pd.DataFrame({
    "id": list(range(1, len(sample["tokens"]) + 1)),
    "token": sample["tokens"],
    "upos": sample["upos"],
    "gold_head": sample["head"],
    "gold_deprel": sample["deprel"],
})

Loading Universal Dependencies Vietnamese VTB...


README.md: 0.00B [00:00, ?B/s]

parquet/vi_vtb/dev.parquet:   0%|          | 0.00/382k [00:00<?, ?B/s]

parquet/vi_vtb/test.parquet:   0%|          | 0.00/225k [00:00<?, ?B/s]

parquet/vi_vtb/train.parquet:   0%|          | 0.00/382k [00:00<?, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Available splits: ['dev', 'test', 'train']
Total training sentences available: 1400
Total testing sentences available: 800
Training sentences used: 500
Testing sentences used: 100


,Dependency Relation,Count
8,punct,1109
2,obj,610
0,nsubj,521
1,root,500
19,case,429
9,advmod,404
21,xcomp,304
4,nmod,301
12,conj,256
26,obl,214


,id,token,upos,gold_head,gold_deprel
0,1,Tôi,11,2,nsubj
1,2,nhớ,16,0,root
2,3,lời,0,2,obj
3,4,anh,0,5,clf:det
4,5,chủ tịch,0,8,nsubj
5,6,xã,0,5,nmod
6,7,Bùi Văn Luyến,10,5,appos
7,8,nhắc,16,3,acl
8,9,đi,14,8,flat:redup
9,10,nhắc,16,8,flat:redup


## Simple Left-Head Baseline

In [2]:
# Baseline đơn giản: mỗi token phụ thuộc vào token đứng ngay trước nó.
# Token đầu tiên được xem là root. Đây là baseline yếu nhưng giúp ta có mốc so sánh.
def to_int_heads(heads):
    return [int(head) if head not in (None, "_") else -1 for head in heads]


def baseline_parse(tokens):
    pred_heads = []
    pred_deprels = []

    for i, _ in enumerate(tokens):
        if i == 0:
            pred_heads.append(0)
            pred_deprels.append("root")
        else:
            pred_heads.append(i)
            pred_deprels.append("dep")

    return pred_heads, pred_deprels


def evaluate_parses(gold_heads_list, gold_deprels_list, pred_heads_list, pred_deprels_list):
    total = 0
    correct_head = 0
    correct_labeled = 0
    gold_labels = []
    pred_labels = []

    for gold_heads, gold_deprels, pred_heads, pred_deprels in zip(
        gold_heads_list, gold_deprels_list, pred_heads_list, pred_deprels_list
    ):
        length = min(len(gold_heads), len(pred_heads))
        for i in range(length):
            if gold_heads[i] == -1:
                continue

            total += 1
            head_ok = gold_heads[i] == pred_heads[i]
            label_ok = gold_deprels[i] == pred_deprels[i]

            correct_head += int(head_ok)
            correct_labeled += int(head_ok and label_ok)
            gold_labels.append(gold_deprels[i])
            pred_labels.append(pred_deprels[i])

    uas = correct_head / total if total else 0
    las = correct_labeled / total if total else 0
    return uas, las, gold_labels, pred_labels

baseline_gold_heads = []
baseline_gold_deprels = []
baseline_pred_heads = []
baseline_pred_deprels = []

for example in test_data:
    pred_heads, pred_deprels = baseline_parse(example["tokens"])
    baseline_gold_heads.append(to_int_heads(example["head"]))
    baseline_gold_deprels.append(example["deprel"])
    baseline_pred_heads.append(pred_heads)
    baseline_pred_deprels.append(pred_deprels)

baseline_uas, baseline_las, _, _ = evaluate_parses(
    baseline_gold_heads,
    baseline_gold_deprels,
    baseline_pred_heads,
    baseline_pred_deprels,
)

print(f"Baseline UAS: {baseline_uas:.4f}")
print(f"Baseline LAS: {baseline_las:.4f}")

Baseline UAS: 0.2296
Baseline LAS: 0.0068


## Public Stanza Dependency Parser

### Parsing Pretokenized Vietnamese

In [5]:
import stanza

# Tải và khởi tạo mô hình Stanza tiếng Việt công khai.
# tokenize_pretokenized=True giúp parser dùng đúng token từ UD_Vietnamese-VTB.
stanza.download("vi")
nlp = stanza.Pipeline(
    lang="vi",
    processors="tokenize,pos,lemma,depparse",
    tokenize_pretokenized=True,
    verbose=False,
)


def stanza_parse(tokens):
    doc = nlp([tokens])
    words = doc.sentences[0].words
    pred_heads = [word.head for word in words]
    pred_deprels = [word.deprel for word in words]
    return pred_heads, pred_deprels


def parse_to_dataframe(tokens, heads, deprels):
    rows = []
    for i, (token, head, deprel) in enumerate(zip(tokens, heads, deprels), start=1):
        head_word = "ROOT" if head == 0 else tokens[head - 1]
        rows.append({
            "id": i,
            "token": token,
            "head_id": head,
            "head_word": head_word,
            "deprel": deprel,
        })
    return pd.DataFrame(rows)

print("Stanza pipeline is ready!")

2026-05-26 22:59:55 INFO: Downloaded file to /home/shinenolife/.cache/stanza/1.12.0/resources/resources.json
2026-05-26 22:59:55 INFO: Downloading default packages for language: vi (Vietnamese) ...


2026-05-26 23:00:46 INFO: Downloaded file to /home/shinenolife/.cache/stanza/1.12.0/resources/vi/default.zip
2026-05-26 23:00:48 INFO: Finished downloading models and saved to /home/shinenolife/.cache/stanza/1.12.0/resources


Stanza pipeline is ready!


### Evaluating Stanza

In [6]:
print("Evaluating Stanza Vietnamese dependency parser...")

stanza_gold_heads = []
stanza_gold_deprels = []
stanza_pred_heads = []
stanza_pred_deprels = []

for example in test_data:
    tokens = example["tokens"]
    pred_heads, pred_deprels = stanza_parse(tokens)

    # Nếu tokenization có lệch hiếm gặp, hàm evaluate_parses sẽ chỉ tính phần khớp độ dài.
    stanza_gold_heads.append(to_int_heads(example["head"]))
    stanza_gold_deprels.append(example["deprel"])
    stanza_pred_heads.append(pred_heads)
    stanza_pred_deprels.append(pred_deprels)

stanza_uas, stanza_las, stanza_gold_labels, stanza_pred_labels = evaluate_parses(
    stanza_gold_heads,
    stanza_gold_deprels,
    stanza_pred_heads,
    stanza_pred_deprels,
)

print(f"Stanza UAS: {stanza_uas:.4f}")
print(f"Stanza LAS: {stanza_las:.4f}")

example = test_data[0]
example_heads, example_deprels = stanza_parse(example["tokens"])
parse_to_dataframe(example["tokens"], example_heads, example_deprels)

Evaluating Stanza Vietnamese dependency parser...
Stanza UAS: 0.7569
Stanza LAS: 0.5935


,id,token,head_id,head_word,deprel
0,1,Thanh,2,bắt chuyện,nsubj
1,2,bắt chuyện,0,ROOT,root
2,3,với,4,Hùng,case
3,4,Hùng,2,bắt chuyện,obl:with
4,5,và,6,nói,cc
5,6,nói,2,bắt chuyện,conj
6,7,:,10,trông,punct
7,8,"""",10,trông,punct
8,9,Tôi,10,trông,nsubj
9,10,trông,2,bắt chuyện,parataxis


# Results

In [7]:
# Create a summary DataFrame
summary_data = {
    "Model": ["Simple Left-Head Baseline", "Stanza Vietnamese Dependency Parser"],
    "UAS": [baseline_uas, stanza_uas],
    "LAS": [baseline_las, stanza_las],
}

summary_df = pd.DataFrame(summary_data)
summary_df.set_index("Model", inplace=True)

print("================ MODEL COMPARISON SUMMARY ================")
display(summary_df)

print("\n" + "="*58)
print("Detailed Dependency Label Report for Stanza:")
print("="*58)
print(classification_report(stanza_gold_labels, stanza_pred_labels, zero_division=0))

================ MODEL COMPARISON SUMMARY ================


,UAS,LAS
Model,,
Simple Left-Head Baseline,0.229575,0.006752
Stanza Vietnamese Dependency Parser,0.756921,0.593518



Detailed Dependency Label Report for Stanza:
                   precision    recall  f1-score   support

              acl       0.00      0.00      0.00         7
         acl:subj       0.27      0.25      0.26        16
         acl:tmod       0.67      0.50      0.57         4
         acl:tonp       0.00      0.00      0.00         0
            advcl       0.33      0.14      0.20        29
  advcl:objective       0.75      0.38      0.50         8
           advmod       0.76      0.89      0.82        88
       advmod:adj       0.25      0.03      0.06        31
       advmod:neg       0.93      0.96      0.95        28
             amod       0.48      0.59      0.53        22
            appos       0.00      0.00      0.00         1
       appos:nmod       0.00      0.00      0.00         4
              aux       0.36      0.57      0.44         7
         aux:pass       1.00      1.00      1.00         4
             case       0.82      0.89      0.85        71
         

### Nhận xét
Stanza cho kết quả tốt hơn baseline vì tận dụng mô hình đã huấn luyện trên dữ liệu Universal Dependencies.